# File 01 — Validitas & Reliabilitas
**EHSL-ML+ | Disertasi Nanan Soekarna**

File ini menguji apakah item-item kuesioner **konsisten** (reliabilitas) dan **benar-benar mengukur konstruk yang dimaksud** (validitas), sebelum dipakai untuk menghitung MAS/ECI/NREI di File 02.

Mengikuti proposal (bagian Validity and Reliability Testing), ada 2 pengujian utama di file ini:

1. **Reliability Analysis (Cronbach's Alpha)** — diuji untuk **4 konstruk**: VfV, FCNC, ID (dari `data_actor_clean.csv`), dan GR (dari `data_respondent_clean.csv`)
2. **Construct Validity (EFA)** — proposal menyebutkan EFA khusus untuk **model 3-konstruk VfV/FCNC/ID** saja (GR tidak ikut EFA, karena bukan bagian dari "the three-construct model" yang divalidasi strukturnya)

Ambang batas yang dipakai (sesuai Tabel 3.16 & 3.17 proposal):

| Uji | Syarat |
|---|---|
| Cronbach's Alpha | ≥ 0.70 (0.70–0.79 cukup, 0.80–0.89 baik, ≥0.90 sangat baik tapi cek redundansi) |
| KMO | ≥ 0.60, idealnya ≥ 0.70 |
| Bartlett's Test | signifikan p < 0.05 |
| Factor loading | ≥ 0.50 (item di bawah 0.40 jelas lemah) |
| Jumlah faktor | eigenvalue > 1.0, dicek juga scree plot |

**Catatan kecil soal dokumen:** ada 2 tempat di proposal yang menyebut angka ambang factor loading sedikit beda (0.40 di satu bagian, 0.50 di Tabel 3.17) — saya pakai 0.50 sebagai acuan utama (itu yang eksplisit disebut "commonly used minimum"), dan 0.40 sebagai batas bawah mutlak. Ada baiknya ini diseragamkan di draf final proposal.

In [1]:
# ---------------------------------------------------------------------------
# SHIM KOMPATIBILITAS (bukan bagian analisis, tidak mengubah hasil apa pun).
# factor_analyzer 0.5.1 -- rilis TERAKHIR yang tersedia -- masih memanggil
# sklearn check_array(force_all_finite=...). Argumen itu DIHAPUS di
# scikit-learn >= 1.8 (diganti ensure_all_finite), sehingga tanpa shim ini
# File 01 gagal dengan TypeError di lingkungan sklearn baru.
# Di sklearn lama (mis. Google Colab) blok ini otomatis TIDAK melakukan apa-apa,
# jadi notebook tetap jalan di kedua lingkungan.
import inspect as _inspect
import sklearn.utils as _sku
import sklearn.utils.validation as _skv

if "force_all_finite" not in _inspect.signature(_skv.check_array).parameters:
    _orig_check_array = _skv.check_array

    def _check_array_compat(*args, **kwargs):
        if "force_all_finite" in kwargs:
            kwargs["ensure_all_finite"] = kwargs.pop("force_all_finite")
        return _orig_check_array(*args, **kwargs)

    _skv.check_array = _check_array_compat
    _sku.check_array = _check_array_compat
    import factor_analyzer.factor_analyzer as _fa1
    import factor_analyzer.confirmatory_factor_analyzer as _fa2
    _fa1.check_array = _check_array_compat
    _fa2.check_array = _check_array_compat
    print("(shim aktif: force_all_finite -> ensure_all_finite untuk factor_analyzer)")
# ---------------------------------------------------------------------------

import numpy as np
import pandas as pd
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity
from factor_analyzer import FactorAnalyzer

df_actor = pd.read_csv("data_actor_clean.csv")
df_resp = pd.read_csv("data_respondent_clean.csv")

item_groups = {
    "VfV":  ["VfV1", "VfV2", "VfV3", "VfV4", "VfV5"],
    "FCNC": ["FCNC1", "FCNC2", "FCNC3", "FCNC4", "FCNC5"],
    "ID":   ["ID1", "ID2", "ID3", "ID4", "ID5"],
}
gr_items = ["GR1", "GR2", "GR3", "GR4", "GR5"]
all_items_efa = [it for g in item_groups.values() for it in g]

print(f"df_actor: {df_actor.shape[0]} baris | df_respondent: {df_resp.shape[0]} baris")

(shim aktif: force_all_finite -> ensure_all_finite untuk factor_analyzer)
df_actor: 896 baris | df_respondent: 199 baris


## 1. Reliability Analysis — Cronbach's Alpha

Analoginya: Cronbach's alpha mengukur "apakah orang yang menjawab tinggi di item 1 juga cenderung menjawab tinggi di item 2, 3, 4, 5 dalam konstruk yang sama". Kalau iya konsisten, alpha tinggi — item-item itu memang mengukur hal yang sama.

Perhitungan alpha di sini pakai **listwise deletion** per konstruk (baris yang punya missing di salah satu item konstruk tsb dikeluarkan dulu khusus untuk hitung alpha) — ingat, item mentah (`VfV1` dst.) memang masih ada yang kosong dari File 00 (lihat pembahasan sebelumnya soal `GR3`).

Dua diagnostik tambahan dihitung per item:
- **Item-total correlation**: seberapa item itu selaras dengan total skor item-item lain di konstruk yang sama. Di bawah 0.30 → lemah, dicurigai tidak benar-benar mengukur hal yang sama.
- **Alpha-if-item-deleted**: alpha yang akan didapat KALAU item itu dibuang. Kalau angka ini lebih tinggi dari alpha keseluruhan, artinya item itu justru "merusak" konsistensi konstruknya.

In [2]:
def cronbach_alpha(df_items):
    d = df_items.dropna()
    k = d.shape[1]
    item_vars = d.var(axis=0, ddof=1)
    total_var = d.sum(axis=1).var(ddof=1)
    alpha = (k / (k - 1)) * (1 - item_vars.sum() / total_var)
    return alpha, len(d)

def interpret_alpha(a):
    if a < 0.60: return "Poor"
    if a < 0.70: return "Questionable"
    if a < 0.80: return "Acceptable"
    if a < 0.90: return "Good"
    return "Excellent (cek redundansi item)"

def item_total_correlation(df_items):
    d = df_items.dropna()
    out = {}
    for col in d.columns:
        rest = d.drop(columns=[col]).sum(axis=1)
        out[col] = d[col].corr(rest)
    return out

def alpha_if_deleted(df_items):
    d = df_items.dropna()
    out = {}
    for col in d.columns:
        a, _ = cronbach_alpha(d.drop(columns=[col]))
        out[col] = a
    return out

reliability_summary = []
for construct, items in item_groups.items():
    alpha, n = cronbach_alpha(df_actor[items])
    reliability_summary.append({"construct": construct, "alpha": alpha, "n": n, "interpretation": interpret_alpha(alpha)})
    print(f"alpha_{construct} = {alpha:.3f}  (n={n}, {interpret_alpha(alpha)})")
    itc = item_total_correlation(df_actor[items])
    aid = alpha_if_deleted(df_actor[items])
    for it in items:
        flag = "  <-- item-total corr < 0.30, PERLU DIREVIEW" if itc[it] < 0.30 else ""
        print(f"    {it}: item-total corr={itc[it]:.3f}  alpha-if-deleted={aid[it]:.3f}{flag}")
    print()

alpha_gr, n_gr = cronbach_alpha(df_resp[gr_items])
reliability_summary.append({"construct": "GR", "alpha": alpha_gr, "n": n_gr, "interpretation": interpret_alpha(alpha_gr)})
print(f"alpha_GR = {alpha_gr:.3f}  (n={n_gr}, {interpret_alpha(alpha_gr)})")
itc_gr = item_total_correlation(df_resp[gr_items])
aid_gr = alpha_if_deleted(df_resp[gr_items])
for it in gr_items:
    flag = "  <-- item-total corr < 0.30, PERLU DIREVIEW" if itc_gr[it] < 0.30 else ""
    print(f"    {it}: item-total corr={itc_gr[it]:.3f}  alpha-if-deleted={aid_gr[it]:.3f}{flag}")

reliability_df = pd.DataFrame(reliability_summary)

alpha_VfV = 0.927  (n=764, Excellent (cek redundansi item))
    VfV1: item-total corr=0.807  alpha-if-deleted=0.911
    VfV2: item-total corr=0.814  alpha-if-deleted=0.910
    VfV3: item-total corr=0.811  alpha-if-deleted=0.910
    VfV4: item-total corr=0.818  alpha-if-deleted=0.909
    VfV5: item-total corr=0.797  alpha-if-deleted=0.913

alpha_FCNC = 0.927  (n=774, Excellent (cek redundansi item))
    FCNC1: item-total corr=0.809  alpha-if-deleted=0.911
    FCNC2: item-total corr=0.809  alpha-if-deleted=0.911
    FCNC3: item-total corr=0.821  alpha-if-deleted=0.908
    FCNC4: item-total corr=0.799  alpha-if-deleted=0.913
    FCNC5: item-total corr=0.807  alpha-if-deleted=0.911

alpha_ID = 0.930  (n=779, Excellent (cek redundansi item))
    ID1: item-total corr=0.802  alpha-if-deleted=0.917
    ID2: item-total corr=0.813  alpha-if-deleted=0.915
    ID3: item-total corr=0.822  alpha-if-deleted=0.913
    ID4: item-total corr=0.825  alpha-if-deleted=0.913
    ID5: item-total corr=0.817  a

## 2. Construct Validity — EFA (khusus VfV + FCNC + ID)

### 2a. Uji kelayakan data (sebelum EFA)

Dua uji ini menjawab pertanyaan: "apakah data ini cocok untuk difaktorkan sama sekali?" Kalau item-itemnya tidak saling berkorelasi secuil pun, EFA tidak ada gunanya.

- **KMO (Kaiser-Meyer-Olkin)**: mengukur proporsi varians antar-item yang mungkin disebabkan faktor bersama. Makin tinggi (mendekati 1), makin cocok untuk EFA.
- **Bartlett's Test of Sphericity**: menguji apakah matriks korelasi antar-item signifikan berbeda dari matriks identitas (yang berarti item-item saling tidak berkorelasi sama sekali). p < 0.05 → layak lanjut.

**Catatan penting soal N:** EFA di sini dihitung di level baris responden-aktor (bukan level responden), karena unit pengukuran VfV/FCNC/ID memang per-aktor. Konsekuensinya: satu responden bisa menyumbang lebih dari 1 baris (kalau dia menemui beberapa aktor), sehingga baris-baris itu tidak sepenuhnya independen satu sama lain (ada "gaya menjawab" yang mirip dari orang yang sama). ini pendekatan umum untuk desain semacam ini, tapi tetap perlu dicatat sebagai keterbatasan saat pelaporan, bukan cacat fatal.

In [3]:
efa_data = df_actor[all_items_efa].dropna()
print(f"N baris dipakai untuk EFA (listwise, level responden-aktor): {len(efa_data)}")
print(f"(sebagai pembanding: syarat minimum EFA=200, CFA=300 -- proposal §3.4-ish)")

kmo_all, kmo_model = calculate_kmo(efa_data)
chi2, p_bartlett = calculate_bartlett_sphericity(efa_data)
print(f"\nKMO overall = {kmo_model:.3f}  (syarat >= 0.60, idealnya >= 0.70)")
print(f"Bartlett's test: chi2={chi2:.1f}, p={p_bartlett:.5f}  (syarat p < 0.05)")
print("-> LAYAK lanjut ke EFA" if (kmo_model >= 0.60 and p_bartlett < 0.05) else "-> TIDAK layak, cek ulang item")

N baris dipakai untuk EFA (listwise, level responden-aktor): 581
(sebagai pembanding: syarat minimum EFA=200, CFA=300 -- proposal §3.4-ish)

KMO overall = 0.953  (syarat >= 0.60, idealnya >= 0.70)
Bartlett's test: chi2=6936.1, p=0.00000  (syarat p < 0.05)
-> LAYAK lanjut ke EFA


### 2b. Jumlah Faktor & Factor Loading

Jumlah faktor ditentukan lewat **dua** kriteria yang dilaporkan berdampingan:

1. **Eigenvalue > 1.0 (kaidah Kaiser)** — kriteria lama. Mudah dihitung, tapi dikenal cenderung **melebih-lebihkan** jumlah faktor, terutama saat jumlah item banyak. Ini kelemahan yang sering dipersoalkan penguji.
2. **Parallel analysis (Horn, 1965)** — kriteria yang lebih dipertahankan secara metodologis. Caranya: bangkitkan ratusan set data acak berukuran sama, hitung eigenvalue-nya, lalu ambil persentil ke-95. Faktor dipertahankan hanya kalau eigenvalue nyatanya **melampaui** eigenvalue yang bisa muncul dari data acak belaka. Dengan kata lain: "apakah faktor ini lebih kuat daripada kebetulan?"

Kalau keduanya sepakat, kesimpulan jumlah faktor jauh lebih kokoh. Harapannya di sini: **3 faktor**, sesuai VfV/FCNC/ID.

In [4]:
fa_full = FactorAnalyzer(rotation=None, n_factors=len(all_items_efa))
fa_full.fit(efa_data)
ev, _ = fa_full.get_eigenvalues()

# ---------------------------- kriteria 1: Kaiser (eigenvalue > 1)
n_kaiser = int((ev > 1.0).sum())

# ---------------------------- kriteria 2: parallel analysis (Horn, 1965)
# Bandingkan eigenvalue nyata dgn eigenvalue dari data ACAK berukuran sama.
# Faktor dipertahankan hanya jika melampaui persentil 95 data acak.
def parallel_analysis(n_obs, n_vars, n_iter=500, persentil=95, seed=42):
    rng = np.random.default_rng(seed)
    ev_acak = np.empty((n_iter, n_vars))
    for i in range(n_iter):
        sim = rng.normal(size=(n_obs, n_vars))
        ev_acak[i] = np.linalg.eigvalsh(np.corrcoef(sim, rowvar=False))[::-1]
    return np.percentile(ev_acak, persentil, axis=0)

ev_ambang = parallel_analysis(efa_data.shape[0], efa_data.shape[1])
n_parallel = int((ev > ev_ambang).sum())

print("Perbandingan eigenvalue nyata vs ambang parallel analysis (persentil 95 data acak):")
print(f"{'faktor':>7} {'ev_nyata':>10} {'ambang_acak':>12}  keputusan")
for i in range(min(6, len(ev))):
    tahan = "TAHAN" if ev[i] > ev_ambang[i] else "buang"
    print(f"{i+1:>7} {ev[i]:>10.3f} {ev_ambang[i]:>12.3f}  {tahan}")

print(f"\nJumlah faktor menurut Kaiser (ev > 1.0)      : {n_kaiser}")
print(f"Jumlah faktor menurut parallel analysis      : {n_parallel}   <- kriteria utama")
if n_kaiser == n_parallel:
    print(f"-> Kedua kriteria SEPAKAT: {n_parallel} faktor. Kesimpulan kokoh.")
else:
    print(f"-> Kedua kriteria BERBEDA (Kaiser {n_kaiser} vs parallel {n_parallel}).")
    print("   Ikuti parallel analysis; laporkan selisih ini sebagai catatan metodologis.")

n_factors_suggested = n_parallel
print(f"\nEigenvalues (5 pertama): {np.round(ev[:5], 2)}")
print(f"Variance explained oleh {n_factors_suggested} faktor pertama: "
      f"{(ev[:n_factors_suggested].sum()/ev.sum()*100):.1f}% dari total varians")

Perbandingan eigenvalue nyata vs ambang parallel analysis (persentil 95 data acak):
 faktor   ev_nyata  ambang_acak  keputusan
      1      8.474        1.335  TAHAN
      2      1.645        1.261  TAHAN
      3      1.419        1.207  TAHAN
      4      0.360        1.163  buang
      5      0.341        1.125  buang
      6      0.326        1.087  buang

Jumlah faktor menurut Kaiser (ev > 1.0)      : 3
Jumlah faktor menurut parallel analysis      : 3   <- kriteria utama
-> Kedua kriteria SEPAKAT: 3 faktor. Kesimpulan kokoh.

Eigenvalues (5 pertama): [8.47 1.65 1.42 0.36 0.34]
Variance explained oleh 3 faktor pertama: 76.9% dari total varians


In [5]:
# ROTASI OBLIQUE (promax), bukan varimax.
# Alasannya: varimax memaksa faktor SALING TIDAK BERKORELASI. Untuk konstruk
# psikososial seperti VfV/FCNC/ID -- yang secara teori memang saling
# berhubungan -- asumsi itu tidak realistis, dan akibatnya cross-loading yang
# muncul sebagian hanyalah artefak rotasi, bukan sifat itemnya.
fa3 = FactorAnalyzer(rotation="promax", n_factors=3)
fa3.fit(efa_data)
loadings = pd.DataFrame(fa3.loadings_, index=all_items_efa,
                        columns=["Factor1", "Factor2", "Factor3"])

print("Factor loadings (rotasi PROMAX / oblique):\n")
print(loadings.round(3).to_string())

# Rotasi oblique memberi bonus: korelasi ANTAR-FAKTOR bisa dibaca langsung.
# Ini justru informasi yang hilang kalau pakai varimax.
if getattr(fa3, "phi_", None) is not None:
    phi = pd.DataFrame(fa3.phi_, index=loadings.columns, columns=loadings.columns)
    print("\nKorelasi antar-faktor (phi) -- hanya tersedia pada rotasi oblique:")
    print(phi.round(3).to_string())
    off = phi.values[np.triu_indices_from(phi.values, k=1)]
    print(f"\nKorelasi antar-faktor berkisar {off.min():.3f} s/d {off.max():.3f}.")
    if np.abs(off).max() > 0.20:
        print("-> Faktor memang saling berkorelasi: rotasi oblique TEPAT, varimax")
        print("   akan menyesatkan karena memaksakan korelasi = 0.")
    else:
        print("-> Korelasi antar-faktor kecil; varimax & promax akan mirip hasilnya.")

print("\n--- Interpretasi otomatis ---")
for item in loadings.index:
    row = loadings.loc[item]
    max_factor = row.abs().idxmax()
    max_load = row[max_factor]
    second_load = row.drop(max_factor).abs().max()
    cross_loading = (second_load >= 0.40) and (abs(max_load) - second_load < 0.20)
    status = "OK"
    if abs(max_load) < 0.40:
        status = "LEMAH (loading < 0.40) -- pertimbangkan dibuang"
    elif abs(max_load) < 0.50:
        status = "BORDERLINE (0.40-0.50) -- perlu direview"
    elif cross_loading:
        status = f"CROSS-LOADING dgn {second_load:.2f} di faktor lain -- perlu direview"
    print(f"{item:6s} -> loading tertinggi {max_load:+.3f} di {max_factor:8s} | {status}")

# ---------------------------------------------------------------------------
# EFA PEMBANDING pada SATU aktor saja (mengatasi pseudo-replikasi).
# Di analisis utama, 1 responden menyumbang sampai 5 baris (satu per aktor yg
# ditemui), sehingga 896 baris itu BUKAN 896 orang independen. Di sini EFA
# diulang hanya pada penilaian terhadap Investigator -- tiap orang tepat 1
# baris -- sebagai bukti bahwa struktur 3 faktor bukan artefak pengulangan.
print("\n" + "="*70)
print("EFA PEMBANDING: hanya penilaian terhadap Investigator (1 baris/orang)")
print("="*70)
solo = df_actor[df_actor["actor_category"] == "Investigator"][all_items_efa].dropna()
print(f"N = {len(solo)} baris = {len(solo)} responden (independen penuh)")
if len(solo) >= 100:
    kmo_solo = calculate_kmo(solo)[1]
    chi2_solo, p_solo = calculate_bartlett_sphericity(solo)
    ev_solo, _ = FactorAnalyzer(rotation=None, n_factors=len(all_items_efa)).fit(solo).get_eigenvalues()
    amb_solo = parallel_analysis(solo.shape[0], solo.shape[1])
    n_solo = int((ev_solo > amb_solo).sum())
    fa_solo = FactorAnalyzer(rotation="promax", n_factors=3).fit(solo)
    load_solo = pd.DataFrame(fa_solo.loadings_, index=all_items_efa,
                             columns=["F1","F2","F3"])
    print(f"KMO = {kmo_solo:.3f} | Bartlett p = {p_solo:.5f}")
    print(f"Jumlah faktor (parallel analysis) = {n_solo}  (analisis utama: {n_parallel})")
    benar = sum(1 for it in load_solo.index if load_solo.loc[it].abs().max() >= 0.50)
    print(f"Item dgn loading >= 0.50 di satu faktor: {benar}/{len(load_solo)}")
    if n_solo == n_parallel:
        print("-> KONSISTEN dgn analisis utama: struktur faktor BUKAN artefak")
        print("   pseudo-replikasi. Ini yang dilaporkan untuk menjawab keberatan penguji.")
    else:
        print("-> BERBEDA dari analisis utama. Laporkan apa adanya sebagai keterbatasan:")
        print("   struktur faktor bisa dipengaruhi non-independensi observasi.")
else:
    print("N terlalu kecil untuk EFA pembanding yang bermakna.")

Factor loadings (rotasi PROMAX / oblique):

       Factor1  Factor2  Factor3
VfV1     0.863    0.033   -0.052
VfV2     0.810   -0.010    0.063
VfV3     0.827   -0.025    0.051
VfV4     0.840    0.038   -0.004
VfV5     0.811    0.008    0.012
FCNC1    0.036   -0.051    0.860
FCNC2    0.020    0.037    0.808
FCNC3   -0.012    0.016    0.849
FCNC4   -0.017    0.061    0.792
FCNC5    0.043    0.020    0.799
ID1     -0.024    0.830    0.033
ID2      0.004    0.830    0.016
ID3     -0.009    0.826    0.031
ID4      0.032    0.850   -0.020
ID5      0.048    0.805    0.008

Korelasi antar-faktor (phi) -- hanya tersedia pada rotasi oblique:
         Factor1  Factor2  Factor3
Factor1    1.000    0.597    0.644
Factor2    0.597    1.000    0.638
Factor3    0.644    0.638    1.000

Korelasi antar-faktor berkisar 0.597 s/d 0.644.
-> Faktor memang saling berkorelasi: rotasi oblique TEPAT, varimax
   akan menyesatkan karena memaksakan korelasi = 0.

--- Interpretasi otomatis ---
VfV1   -> loading ter

KMO = 0.931 | Bartlett p = 0.00000
Jumlah faktor (parallel analysis) = 3  (analisis utama: 3)
Item dgn loading >= 0.50 di satu faktor: 15/15
-> KONSISTEN dgn analisis utama: struktur faktor BUKAN artefak
   pseudo-replikasi. Ini yang dilaporkan untuk menjawab keberatan penguji.


**Cara membaca tabel loading di atas:** tiap item seharusnya loading tinggi (idealnya ≥0.50) di **satu** faktor saja, dan rendah di faktor lain. Kalau VfV1-VfV5 semua loading tinggi di faktor yang sama, itu tandanya kelima item itu memang mengukur konstruk yang sama (VfV) — bagus. Kalau ada item yang loading-nya "nyebar" di 2 faktor sekaligus (cross-loading), itu tandanya item itu ambigu, bisa jadi kalimatnya membingungkan sehingga responden menafsirkannya bisa mengarah ke 2 konstruk berbeda.

**Tambahan dari rotasi promax:** tabel korelasi antar-faktor (phi) di atas adalah informasi yang **tidak bisa didapat dari varimax**. Kalau korelasi antar-faktor ternyata besar, itu bukti empiris bahwa VfV/FCNC/ID memang konstruk yang berkaitan — temuan substantif yang layak dilaporkan, bukan sekadar detail teknis.

**Soal EFA pembanding satu aktor:** ini menjawab langsung keberatan metodologis paling mungkin dari penguji — bahwa 896 baris bukan 896 orang independen. Kalau struktur 3 faktor tetap muncul pada 196 responden yang benar-benar independen, keberatan itu terjawab dengan data, bukan dengan catatan kaki.

## 3. Item Reduction — Ringkasan & Rekomendasi

**Penting:** proposal eksplisit bilang *"Item reduction must not be based only on statistical results; theoretical relevance and expert judgment are also considered."* Jadi bagian ini hanya **melaporkan** item yang perlu direview — bukan otomatis membuang item. Keputusan akhir tetap di tangan Anda (dan idealnya expert panel), bukan di tangan skrip ini.

In [6]:
print("=== RINGKASAN RELIABILITAS ===")
print(reliability_df.to_string(index=False))

print("\n=== ITEM YANG DIREKOMENDASIKAN UNTUK DIREVIEW ===")
review_list = []
for construct, items in item_groups.items():
    itc = item_total_correlation(df_actor[items])
    for it in items:
        if itc[it] < 0.30:
            review_list.append({"item": it, "construct": construct, "alasan": f"item-total corr rendah ({itc[it]:.3f})"})
itc_gr = item_total_correlation(df_resp[gr_items])
for it in gr_items:
    if itc_gr[it] < 0.30:
        review_list.append({"item": it, "construct": "GR", "alasan": f"item-total corr rendah ({itc_gr[it]:.3f})"})

for item in loadings.index:
    row = loadings.loc[item]
    max_load = row.max()
    if max_load < 0.50:
        review_list.append({"item": item, "construct": "?", "alasan": f"factor loading rendah ({max_load:.3f})"})

if review_list:
    review_df = pd.DataFrame(review_list)
    print(review_df.to_string(index=False))
else:
    print("Tidak ada item yang perlu direview -- semua di atas ambang batas.")
    review_df = pd.DataFrame(columns=["item","construct","alasan"])

review_df.to_csv("item_reduction_report.csv", index=False)
reliability_df.to_csv("reliability_summary.csv", index=False)
print("\nDisimpan: item_reduction_report.csv, reliability_summary.csv")

=== RINGKASAN RELIABILITAS ===
construct    alpha   n                  interpretation
      VfV 0.927420 764 Excellent (cek redundansi item)
     FCNC 0.927338 774 Excellent (cek redundansi item)
       ID 0.930382 779 Excellent (cek redundansi item)
       GR 0.730633 166                      Acceptable

=== ITEM YANG DIREKOMENDASIKAN UNTUK DIREVIEW ===


Tidak ada item yang perlu direview -- semua di atas ambang batas.

Disimpan: item_reduction_report.csv, reliability_summary.csv


## Ringkasan File 01

| Konstruk | Cronbach's Alpha | EFA |
|---|---|---|
| VfV | lihat output di atas | loading bersih di 1 faktor (promax) |
| FCNC | lihat output di atas | loading bersih di 1 faktor (promax) |
| ID | lihat output di atas | loading bersih di 1 faktor (promax) |
| GR | lihat output di atas | tidak diuji EFA (di luar model 3-konstruk) |

**Tiga perbaikan metodologis di file ini:**
1. **Rotasi promax (oblique)** menggantikan varimax — karena VfV/FCNC/ID secara teori saling berkorelasi, dan memaksakan korelasi nol membuat cross-loading tampak lebih buruk dari kenyataannya. Bonusnya: korelasi antar-faktor kini bisa dilaporkan.
2. **Parallel analysis** dilaporkan berdampingan dengan kaidah Kaiser — kriteria yang lebih dipertahankan di sidang, karena eigenvalue > 1 dikenal cenderung melebih-lebihkan jumlah faktor.
3. **EFA pembanding pada satu aktor** (1 baris per orang) — bukti empiris bahwa struktur 3 faktor bukan artefak pseudo-replikasi, menjawab keberatan non-independensi observasi dengan data.

**Catatan jujur soal data dummy ini:** alpha VfV/FCNC/ID kemungkinan keluar sangat tinggi (>0.90) karena cara data dummy dibangkitkan membuat item dalam 1 konstruk terlalu mirip. Di data riil nanti, alpha 0.70-0.85 justru lebih realistis. Yang penting dari latihan ini bukan angka persisnya, tapi **caranya**.

**Output yang dibawa ke File 02:** `data_actor_clean.csv` dan `data_respondent_clean.csv` (tidak berubah — belum ada item yang dibuang, karena keputusan itu masih perlu Anda tinjau lewat `item_reduction_report.csv`), plus `reliability_summary.csv`.